# 贏球估計

先用瀏覽器打開同資料夾的 `委託.html` 讀委託。這一本由你跟 AI 一起寫。一次請 AI 完成下面一格程式，跑過、看懂，再進行下一格。最後一節只有你寫。

## 1 資料


In [ ]:
# 讀 data/mlb_games.csv，印出筆數、欄位。看同一場比賽編號是不是有兩列。
from pathlib import Path

import pandas as pd

try:
    here = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    here = Path.cwd()

games = pd.read_csv(here / "data" / "mlb_games.csv")

# 讀進來先清洗欄名（原始資料有的欄名前面帶空白，例如「 安打（支）」）
before = list(games.columns)
games.columns = games.columns.str.strip()
print("清洗前欄名：", before)
print("清洗後欄名：", list(games.columns), "\n")

print("筆數 =", len(games), "列；共", games["比賽編號"].nunique(), "場")
print("\n球季別筆數：")
print(games["球季"].value_counts().sort_index().to_string())

rows_per_game = games.groupby("比賽編號").size()
print("\n每場列數的分布：", rows_per_game.value_counts().to_dict())

print("\n各欄型別：")
print(games.dtypes.to_string())

print("\n有沒有空白值：")
print(games.isna().sum().to_string())

g0 = games["比賽編號"].iloc[0]
print(f"\n第一場（比賽編號 {g0}）的兩列：")
print(games.loc[games["比賽編號"] == g0, ["球隊", "主場（1為主場）", "得分（分）", "安打差（支）", "贏球（1為贏）"]].to_string(index=False))

pair = games.groupby("比賽編號")[["安打差（支）", "贏球（1為贏）"]].sum()
print("\n每場安打差相加全部是 0：", bool((pair["安打差（支）"] == 0).all()))
print("每場贏球相加全部是 1：", bool((pair["贏球（1為贏）"] == 1).all()))


## 2 前處理

- 每場兩列互為鏡像 → 不刪列，改用「換邊會變號」的對稱特徵，一場的兩列各自提供一份證據。
- 用到的欄：`安打差（支）`、`失誤差（次）`（= 對手失誤 − 自己失誤，自己造）、`主場（1為主場）`。
- 不用的欄：`得分（分）`（是輸贏的定義）、`球季`／`日期`／`比賽編號`／`球隊`（識別用，不是成因）、`安打（支）` 與 `對手安打（支）`（已由 `安打差` 代表）、`失誤（次）` 與 `對手失誤（次）`（已由 `失誤差` 代表）。
- 三欄尺度差很大（安打差 ±20 上下、失誤差個位數、主場只有 0/1）→ 標準化。

In [ ]:
# 處理每場兩列、互為鏡像的問題，做出能拿去訓練的樣本。
# 留下贏球，以及你決定要用的欄。
# 各欄尺度不同的話，做出能訓練的特徵，並印出處理後的形狀。
import numpy as np
from sklearn.preprocessing import StandardScaler

df = games.copy()

# 自己造的對稱欄：換邊時會變號（對手失誤 − 自己失誤）
df["失誤差（次）"] = df["對手失誤（次）"] - df["失誤（次）"]

FEATURES = ["安打差（支）", "失誤差（次）", "主場（1為主場）"]

X_raw = df[FEATURES].to_numpy(dtype=float)
y = df["贏球（1為贏）"].to_numpy(dtype=int)

print("原始尺度（各欄差很多）：")
print(df[FEATURES].agg(["min", "max", "mean", "std"]).T.round(2).to_string(), "\n")

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print("標準化後（平均 0、標準差 1）：")
print(pd.DataFrame(X, columns=FEATURES).agg(["min", "max", "mean", "std"]).T.round(3).to_string(), "\n")

print("X 形狀 =", X.shape, " y 形狀 =", y.shape)
print("贏球比例 =", round(y.mean(), 4))

# 鏡像檢查：同一場的兩列，特徵應該互為相反數（主場 1↔0、安打差 ±、失誤差 ±）
g0 = df["比賽編號"].iloc[0]
i0, i1 = np.where(df["比賽編號"].to_numpy() == g0)[0]
print(f"\n比賽 {g0} 的兩列特徵：")
print("  列1:", np.round(X[i0], 3))
print("  列2:", np.round(X[i1], 3))
print("  互為相反數：", bool(np.allclose(X[i0], -X[i1])))


## 3 訓練

- 用第 2 格標準化後的 `X` 和 `y` 做邏輯回歸。
- 印出截距與三個係數，並換算成「勝算比」（odds ratio）。
- 畫一張 `安打差` → 贏球機率的曲線，主場／客場各一條。

In [ ]:
# 用前一格的結果做邏輯回歸，印出係數，畫一張看得出贏球機率怎麼走的圖。
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "DFKai-SB", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

clf = LogisticRegression(max_iter=1000)
clf.fit(X, y)

b = clf.intercept_[0]
w = clf.coef_[0]

print("截距 b =", round(b, 6))
print()
print("欄位            係數 w      勝算比 e^w")
for name, wi in zip(FEATURES, w):
    print(f"{name:<14}{wi:>9.4f}{np.exp(wi):>12.4f}")
print()
print("訓練準確率 =", round(accuracy_score(y, clf.predict(X)), 4))

# 把「原始安打差」換回標準化尺規，才能代進模型
def std_of(col_index, raw):
    return (raw - scaler.mean_[col_index]) / scaler.scale_[col_index]

hd = np.linspace(-12, 12, 241)
z_hit = std_of(0, hd)
z_err = std_of(1, 0)      # 失誤差固定在平均值
z_home = std_of(2, 1)     # 主場
z_away = std_of(2, 0)     # 客場

def prob(z_place):
    z = b + w[0] * z_hit + w[1] * z_err + w[2] * z_place
    return 1 / (1 + np.exp(-z))

p_home = prob(z_home)
p_away = prob(z_away)

# 對照實際資料：主場球隊依安打差分箱後的真實勝率
home_rows = df[df["主場（1為主場）"] == 1]
cut = pd.cut(home_rows["安打差（支）"], bins=np.arange(-12, 13, 2))
emp = home_rows.groupby(cut, observed=True)["贏球（1為贏）"].agg(["mean", "size"])
emp_x = [iv.mid for iv in emp.index]
emp_n = emp["size"].to_numpy()
emp_p = emp["mean"].to_numpy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hd, p_home, lw=2.2, label="模型：主場球隊")
ax.plot(hd, p_away, lw=2.2, ls="--", label="模型：客隊")
ax.scatter(emp_x, emp_p, s=np.clip(emp_n * 1.5, 20, 220), color="black",
           alpha=0.55, zorder=3, label="實際勝率（點越大樣本越多）")
ax.axhline(0.5, color="gray", lw=0.8, ls=":")
ax.axvline(0, color="gray", lw=0.8, ls=":")
ax.set_xlabel("安打差（支，己方減對手）")
ax.set_ylabel("贏球機率")
ax.set_title("安打差與贏球機率（失誤差固定在平均）")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# 同一場比賽兩隊的安打差互為相反數，所以驗「機率相加 = 1」要把客隊曲線左右翻轉
print()
print("同一場比賽的兩隊（主隊安打差 +h，客隊就是 -h）：")
for v in (-10, -5, -3, 0, 3, 5, 10):
    k = np.abs(hd - v).argmin()
    km = np.abs(hd + v).argmin()
    print(f"  安打差 {v:>3}：主場 {p_home[k]:.4f} + 客隊 {p_away[km]:.4f} = {p_home[k] + p_away[km]:.4f}")

print()
print("同樣安打差，主場 vs 客場的差別：")
for v in (-3, 0, 3, 5):
    k = np.abs(hd - v).argmin()
    print(f"  安打差 {v:>3}：主場 {p_home[k]:.4f}，客場 {p_away[k]:.4f}，差 {p_home[k] - p_away[k]:+.4f}")
